<a id="lp-overview"></a>

$$
\text{线性规划 (Linear Programming, LP): 目标函数和约束条件都是线性的优化问题} \\ \quad \\
\text{一般形式:} \\
\max_{x} \ z = c^T x \quad \text{s.t.} \quad A x \leq b, \ x \geq 0 \\
x \in \mathbb{R}^n \text{: 决策变量 (Decision Variable)}, \quad c \text{: 价值系数}, \quad A \in \mathbb{R}^{m \times n}, b \text{: 资源约束} \\ \quad \\
\text{例 (生产计划 Production Planning): 工厂生产两种产品, 每件利润 3 和 5} \\
\text{车间 1 每周最多 4 小时, 只做产品 1, 每件 1 小时} \\
\text{车间 2 每周最多 12 小时, 只做产品 2, 每件 2 小时} \\
\text{车间 3 每周最多 18 小时, 产品 1 每件 3 小时, 产品 2 每件 2 小时} \\ \quad \\
\max \ z = 3 x_1 + 5 x_2 \\
\text{s.t.} \quad x_1 \leq 4, \quad 2 x_2 \leq 12, \quad 3 x_1 + 2 x_2 \leq 18, \quad x_1, x_2 \geq 0 \\ \quad \\
\text{其他形式都可以化成这个形式:} \\
\min c^T x \iff \max (-c)^T x, \quad a^T x \geq b \iff -a^T x \leq -b, \quad a^T x = b \iff a^T x \leq b \text{ 且 } -a^T x \leq -b
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
from scipy.optimize import linprog

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

c = np.array([3.0, 5.0])
A = np.array([[1.0, 0.0],
              [0.0, 2.0],
              [3.0, 2.0]])
b = np.array([4.0, 12.0, 18.0])

# linprog 默认求最小值, 所以传入 -c
res = linprog(-c, A_ub=A, b_ub=b, bounds=[(0, None)] * 2, method='highs')
print(f'linprog: x = {res.x}, z = {-res.fun:.4f}')

<a id="graphical-method"></a>

$$
\text{图解法 (Graphical Method): 只有两个变量时可以画出来} \\ \quad \\
\text{1. 可行域 (Feasible Region): 每个约束 } a_i^T x \leq b_i \text{ 是一个半平面, 可行域是所有半平面的交集, 是一个凸多边形} \\
\text{凸集 (Convex Set) 的证明: 设 x, y 都可行, } 0 \leq \lambda \leq 1 \\
A(\lambda x + (1 - \lambda) y) = \lambda A x + (1 - \lambda) A y \leq \lambda b + (1 - \lambda) b = b \\
\text{所以两点连线上的点也都可行} \\ \quad \\
\text{2. 目标函数的等值线 } c^T x = z \text{ 是一族平行直线, z 增大时沿 c 的方向平移, 平移到最后一个还和可行域相交的位置} \\ \quad \\
\text{3. 为什么最优解一定在顶点 (Vertex) 上:} \\
\text{在线段 } x(\lambda) = \lambda x + (1 - \lambda) y \text{ 上, } c^T x(\lambda) = \lambda c^T x + (1 - \lambda) c^T y \text{ 是 } \lambda \text{ 的线性函数} \\
\text{线性函数在区间 [0, 1] 上的最大值在端点处取得} \\
\text{凸多边形内任意一点都在某条线段上, 线段的端点在边上, 边上的点又在两个顶点之间} \\
\implies \text{最优值一定可以在某个顶点取到 (如果 c 平行于某条边, 整条边都是最优解)} \\ \quad \\
\text{所以一个 (很笨的) 解法是: 求出所有顶点, 逐个比较目标函数值}
$$

In [ ]:
# 所有约束写成 G x <= h (包括 x >= 0, 即 -x <= 0)
G = np.vstack([A, -np.eye(2)])
h = np.concatenate([b, np.zeros(2)])

vertices = []
for i, j in combinations(range(len(G)), 2):
    M = G[[i, j]]
    if abs(np.linalg.det(M)) < 1e-12:      # 两条线平行, 没有交点
        continue
    v = np.linalg.solve(M, h[[i, j]])      # 两条边界线的交点
    if np.all(G @ v <= h + 1e-9):          # 只保留可行的交点
        vertices.append(v)
vertices = np.unique(np.round(vertices, 9), axis=0) + 0.0   # + 0.0 把 -0. 变成 0.
values = vertices @ c
for v, z in zip(vertices, values):
    print(f'顶点 {v}: z = {z:.1f}')
best = vertices[np.argmax(values)]
print(f'枚举顶点的最优解 = {best}, z = {values.max():.1f}, 和 linprog 一致? {np.allclose(best, res.x)}')

# 可行域 + 目标函数等值线
xs, ys = np.meshgrid(np.linspace(0, 7, 400), np.linspace(0, 9, 400))
feasible = np.all(np.tensordot(G, np.stack([xs, ys]), axes=1) <= h[:, None, None], axis=0)
order = np.argsort(np.arctan2(vertices[:, 1] - vertices[:, 1].mean(), vertices[:, 0] - vertices[:, 0].mean()))

plt.figure(figsize=(4, 3))
plt.contourf(xs, ys, feasible, levels=[0.5, 1.5], colors=['#cce5ff'])
plt.fill(*vertices[order].T, facecolor='none', edgecolor='b', label='feasible region')
x_line = np.linspace(0, 7, 10)
for z in [12, 24, 36]:
    plt.plot(x_line, (z - 3 * x_line) / 5, '--', label=f'3x1 + 5x2 = {z}')
plt.plot(*best, 'ro', label=f'optimum {best}')
plt.xlim(0, 7)
plt.ylim(0, 9)
plt.xlabel('x1')
plt.ylabel('x2')
plot_nicely()
plt.show()

<a id="slack"></a>

$$
\text{松弛变量 (Slack Variable): 把不等式变成等式} \\
a_i^T x \leq b_i \iff a_i^T x + s_i = b_i, \ s_i \geq 0 \quad (s_i \text{ 是第 i 种资源剩余的量}) \\
\begin{bmatrix} A & I \end{bmatrix} \begin{bmatrix} x \\ s \end{bmatrix} = b, \quad x, s \geq 0 \\ \quad \\
\text{现在有 m 个方程, n + m 个未知数. 基本解 (Basic Solution):} \\
\text{选 m 个变量作为基变量 (Basic Variable), 其余 n 个非基变量 (Non-basic) 设为 0, 解 m 元线性方程组 } B x_B = b \\
\text{若解出来 } x_B \geq 0 \text{, 叫基本可行解 (Basic Feasible Solution, BFS)} \\ \quad \\
\text{几何意义: 非基变量为 0 = 对应的约束取等号 = 落在 n 条边界线上 = 顶点} \\
\text{所以 BFS} \iff \text{顶点, 最多 } \binom{n + m}{m} \text{ 个}
$$

In [ ]:
A_eq = np.hstack([A, np.eye(3)])    # [A | I], 变量顺序: x1, x2, s1, s2, s3
names = ['x1', 'x2', 's1', 's2', 's3']
for basis in combinations(range(5), 3):
    B = A_eq[:, basis]
    if abs(np.linalg.det(B)) < 1e-12:
        continue
    x_full = np.zeros(5)
    x_full[list(basis)] = np.linalg.solve(B, b)
    tag = '可行' if np.all(x_full >= -1e-9) else '不可行'
    if tag == '可行':
        print(f'基 {[names[k] for k in basis]}: (x1, x2) = {x_full[:2]}, s = {x_full[2:]}, z = {c @ x_full[:2]:.1f}')

<a id="simplex"></a>

$$
\text{单纯形法 (Simplex Method): 从一个顶点出发, 每次沿一条边走到目标函数更大的相邻顶点} \\ \quad \\
\text{把目标函数也写成一行方程: } z - c^T x = 0 \text{, 单纯形表 (Tableau):} \\
\begin{array}{c|cc|c}
 & x & s & \text{RHS} \\
\hline
\text{约束} & A & I & b \\
\hline
z & -c^T & 0 & 0
\end{array} \\
\text{初始基 = 松弛变量 (x = 0, s = b, 即原点, 要求 b} \geq 0 \text{)} \\ \quad \\
\text{1. 最优性判断: 设最后一行 (检验数 reduced cost) 是 } r \text{, 它表示 } z + \sum_{j \in \text{非基}} r_j x_j = z_0 \\
z = z_0 - \sum_{j} r_j x_j \text{, 非基变量当前都是 0} \\
\text{若所有 } r_j \geq 0 \text{: 任何非基变量增大都只会让 z 减小} \implies \text{当前就是最优解} \\
\text{若有 } r_e < 0 \text{: 让 } x_e \text{ 从 0 增大, z 就会增大, } x_e \text{ 作为进基变量 (Entering Variable), 通常选最负的那一列} \\ \quad \\
\text{2. 比值检验 (Ratio Test): } x_e \text{ 最多能增加多少?} \\
\text{约束第 i 行当前是 } x_{B_i} + \bar{a}_{ie} x_e = \bar{b}_i \text{ (其他非基变量为 0)} \implies x_{B_i} = \bar{b}_i - \bar{a}_{ie} x_e \\
\text{要保持 } x_{B_i} \geq 0 \text{: 若 } \bar{a}_{ie} > 0 \text{, 需要 } x_e \leq \frac{\bar{b}_i}{\bar{a}_{ie}} \text{; 若 } \bar{a}_{ie} \leq 0 \text{, } x_e \text{ 增大不会让 } x_{B_i} \text{ 变负, 没有限制} \\
t^* = \min_{i: \bar{a}_{ie} > 0} \frac{\bar{b}_i}{\bar{a}_{ie}} \text{, 取到最小值的那一行的基变量最先变成 0, 作为出基变量 (Leaving Variable)} \\
\text{(若这一列没有正数: } x_e \text{ 可以无限增大, 问题无界 unbounded)} \\ \quad \\
\text{3. 转轴 (Pivot): 用高斯消元把进基列变成单位向量 (主元行除以主元, 其他行减去主元行的倍数)} \\
\text{这正是在新的基下重新解方程组, 目标函数行也一起消元, 新的 RHS 角上就是新的 z}
$$

In [ ]:
def simplex_max(c, A, b, verbose=True):
    # max c^T x, s.t. A x <= b, x >= 0, 要求 b >= 0 (原点可行)
    m, n = A.shape
    T = np.zeros((m + 1, n + m + 1))
    T[:m, :n] = A
    T[:m, n:n + m] = np.eye(m)
    T[:m, -1] = b
    T[-1, :n] = -c
    basis = list(range(n, n + m))
    it = 0
    while True:
        col = int(np.argmin(T[-1, :-1]))
        if T[-1, col] >= -1e-12:                     # 检验数全部 >= 0: 最优
            break
        positive = T[:m, col] > 1e-12
        if not positive.any():
            raise ValueError('无界 (unbounded)')
        ratios = np.full(m, np.inf)
        ratios[positive] = T[:m, -1][positive] / T[:m, col][positive]
        row = int(np.argmin(ratios))                 # 比值检验
        if verbose:
            print(f'第 {it + 1} 次转轴: 进基 = 第 {col} 列, 比值 = {ratios}, 出基 = 第 {basis[row]} 列 (第 {row} 行)')
        T[row] /= T[row, col]
        for i in range(m + 1):
            if i != row:
                T[i] -= T[i, col] * T[row]
        basis[row] = col
        it += 1
        if verbose:
            x_now = np.zeros(n + m); x_now[basis] = T[:m, -1]
            print(f'    当前顶点 x = {x_now[:n]}, z = {T[-1, -1]:.4f}')
    x = np.zeros(n + m)
    x[basis] = T[:m, -1]
    return x[:n], T[-1, -1], T, basis

x_opt, z_opt, T_final, basis_final = simplex_max(c, A, b)
print(f'\n单纯形法: x = {x_opt}, z = {z_opt:.4f}')
print(f'最终单纯形表 = \n{np.round(T_final, 4)}')
print(f'和 linprog 一致? {np.allclose(x_opt, res.x) and np.isclose(z_opt, -res.fun)}')

# 随机测试: 更大的问题
rng = np.random.default_rng(0)
all_ok = True
for _ in range(50):
    m_r, n_r = rng.integers(2, 7, size=2)
    A_r = rng.uniform(0.1, 5, size=(m_r, n_r))
    b_r = rng.uniform(5, 20, size=m_r)
    c_r = rng.uniform(0.5, 5, size=n_r)
    _, z_r, _, _ = simplex_max(c_r, A_r, b_r, verbose=False)
    lp_r = linprog(-c_r, A_ub=A_r, b_ub=b_r, method='highs')
    all_ok &= np.isclose(z_r, -lp_r.fun)
print(f'50 个随机 LP, 手写单纯形法的最优值全部和 linprog 一致? {all_ok}')

<a id="duality"></a>

$$
\text{对偶 (Duality): 用拉格朗日乘子给原问题的最优值找上界} \\ \quad \\
\text{原问题 (Primal): } \max c^T x \ \text{ s.t. } A x \leq b, \ x \geq 0 \\
\text{对每个约束引入乘子 } y_i \geq 0 \text{, 拉格朗日函数 } L(x, y) = c^T x + y^T (b - A x) \\
\text{对任意可行的 x: } b - Ax \geq 0, \ y \geq 0 \implies y^T(b - Ax) \geq 0 \implies c^T x \leq L(x, y) \\
\text{所以 } z^* \leq \max_{x \geq 0} L(x, y) = \max_{x \geq 0} \left[ b^T y + (c - A^T y)^T x \right] \\
\text{若 } (c - A^T y) \text{ 有某个分量 > 0, 让对应的 } x_j \to \infty \text{, 上界是无穷大, 没用} \\
\text{若 } A^T y \geq c \text{, 则 } (c - A^T y)^T x \leq 0 \text{, 取 x = 0 得到最大值, 上界 = } b^T y \\ \quad \\
\text{想要最紧的上界} \implies \text{对偶问题 (Dual): } \min_y b^T y \ \text{ s.t. } A^T y \geq c, \ y \geq 0 \\ \quad \\
\text{弱对偶 (Weak Duality): x 原问题可行, y 对偶问题可行 } \implies c^T x \leq (A^T y)^T x = y^T A x \leq y^T b \\
\text{强对偶 (Strong Duality): 线性规划有最优解时 } c^T x^* = b^T y^* \text{ (证明需要 Farkas 引理, 这里用数值验证)} \\ \quad \\
\text{互补松弛 (Complementary Slackness): 强对偶时上面两个不等号都取等号} \\
y_i^* (b - A x^*)_i = 0 \text{: 资源有剩余 (约束不紧) 的, 对偶价格一定为 0} \\
x_j^* (A^T y^* - c)_j = 0 \text{: 生产了的产品, 它消耗资源的 "价值" 正好等于利润} \\ \quad \\
\text{影子价格 (Shadow Price): } y_i^* = \frac{\partial z^*}{\partial b_i} \text{, 第 i 种资源多一个单位, 最优利润增加多少} \\
\text{和拉格朗日乘子 } \lambda = \frac{df^*}{dc} \text{ 是同一个结论} \\
\text{在单纯形表中: 目标行是 "原目标行 + } y^T \times \text{约束行" (行变换的累积)} \\
\text{松弛变量列原来是 I 和 0, 所以最终表里松弛变量列下的检验数就是 } y^T \text{, x 列下是 } A^T y - c \geq 0 \text{ (对偶可行)}
$$

数学基础: [拉格朗日乘子法与影子价格](../math_basics/calculus.ipynb#lagrange)

In [ ]:
n_var, m_con = A.shape[1], A.shape[0]
y_from_tableau = T_final[-1, n_var:n_var + m_con]
reduced_x = T_final[-1, :n_var]
print(f'单纯形表松弛列的检验数 y = {y_from_tableau}')
print(f'x 列的检验数 = {reduced_x}, A^T y - c = {A.T @ y_from_tableau - c}')

# 直接求解对偶问题 min b^T y, s.t. A^T y >= c, y >= 0
dual = linprog(b, A_ub=-A.T, b_ub=-c, bounds=[(0, None)] * m_con, method='highs')
print(f'对偶问题: y* = {dual.x}, b^T y* = {dual.fun:.4f}, 原问题 z* = {z_opt:.4f} (强对偶)')
print(f'linprog 的 marginals (对 -z 的导数) = {res.ineqlin.marginals + 0.0}, 取负 = {-res.ineqlin.marginals + 0.0}')

# 互补松弛
slack = b - A @ x_opt
print(f'剩余资源 b - Ax* = {slack}, y* (b - Ax*) = {y_from_tableau * slack}')

# 影子价格 = dz*/db_i (有限差分)
for i in range(m_con):
    db = np.zeros(m_con); db[i] = 0.01
    z_plus = -linprog(-c, A_ub=A, b_ub=b + db, method='highs').fun
    print(f'车间 {i + 1} 多 0.01 小时: z* 增加 {z_plus - z_opt:.4f}, (dz*/db) = {(z_plus - z_opt) / 0.01:.4f}, y = {y_from_tableau[i]:.4f}')

# 弱对偶: 任意可行的 x, y
rng = np.random.default_rng(1)
xs_feas = rng.uniform(0, 6, size=(5000, 2))
xs_feas = xs_feas[np.all(xs_feas @ A.T <= b, axis=1)]
ys_feas = rng.uniform(0, 5, size=(5000, 3))
ys_feas = ys_feas[np.all(ys_feas @ A >= c, axis=1)]
print(f'弱对偶: 可行 x 的最大 c^T x = {np.max(xs_feas @ c):.4f} <= 可行 y 的最小 b^T y = {np.min(ys_feas @ b):.4f}')

<a id="diet-problem"></a>

$$
\text{对偶的用处: 营养配餐问题 (Diet Problem)} \\
\min \ c^T x \quad \text{s.t.} \quad A x \geq b, \ x \geq 0 \\
x_j \text{: 第 j 种食物的份数, } c_j \text{: 单价, } A_{ij} \text{: 每份食物 j 含营养 i 的量, } b_i \text{: 营养 i 的最低需求} \\ \quad \\
\text{原点 x = 0 不可行 (营养不够), 上面写的单纯形法不能直接用 (需要两阶段法 / 大 M 法找初始顶点)} \\
\text{但它的对偶问题: } \max b^T y \ \text{ s.t. } A^T y \leq c, \ y \geq 0 \text{, 原点可行, 可以直接用单纯形法} \\
\text{再把对偶问题的影子价格读出来, 就是原问题的最优解 (对偶的对偶 = 原问题)} \\ \quad \\
\text{对偶的经济解释: 一家营养片公司给每种营养定价 } y_i \text{, 要让每种食物的营养总价不超过食物的价格 } (A^T y \leq c) \\
\text{(否则顾客会直接买食物), 在此前提下最大化收入 } b^T y \text{. 强对偶说: 最高收入 = 最低配餐成本}
$$

In [ ]:
foods = ['米饭', '鸡蛋', '牛奶', '青菜']
cost = np.array([1.0, 1.5, 2.5, 0.8])
#                  米饭  鸡蛋 牛奶  青菜
nutrient = np.array([[2.0, 6.0, 8.0, 1.0],     # 蛋白质
                     [30., 1.0, 12., 4.0],     # 碳水
                     [0.0, 0.5, 3.0, 1.5]])    # 钙
need = np.array([40.0, 120.0, 10.0])

# 原问题直接用 linprog
primal = linprog(cost, A_ub=-nutrient, b_ub=-need, bounds=[(0, None)] * 4, method='highs')
# 对偶问题用手写的单纯形法: max need^T y, s.t. nutrient^T y <= cost
y_diet, rev, T_diet, _ = simplex_max(need, nutrient.T, cost, verbose=False)
x_from_dual = T_diet[-1, len(need):len(need) + len(cost)]   # 对偶问题松弛列的检验数 = 原问题的解

print(f'linprog 原问题: x = {np.round(primal.x, 4)}, 最低成本 = {primal.fun:.4f}')
print(f'单纯形法解对偶: 营养定价 y = {np.round(y_diet, 4)}, 最高收入 = {rev:.4f}')
print(f'从对偶的单纯形表读出的原问题解 x = {np.round(x_from_dual, 4)}, 一致? {np.allclose(x_from_dual, primal.x, atol=1e-8)}')
for f, q in zip(foods, x_from_dual):
    print(f'  {f}: {q:.4f} 份')
print(f'实际营养 = {nutrient @ x_from_dual}, 需求 = {need}')